In [5]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")       # charge TAVILY_API_KEY dans les variables d'environnement

llm = ChatOllama(model="qwen3:4b", temperature=0, reasoning=True)

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

duckduckgo = DuckDuckGoSearchRun(
    api_wrapper=DuckDuckGoSearchAPIWrapper(backend="duckduckgo", max_results=3),
    handle_tool_error=True,   # en cas d'erreur, l'agent reçoit un message au lieu de planter
)
print(duckduckgo.name, ":", duckduckgo.description[:100], "...")
print()
print(duckduckgo.invoke("capitale du Maroc population")[:500])

duckduckgo_search : A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current event ...



DDGSException: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=capitale%20du%20Maroc%20population) > error resolving DNS > DNS error: no records found for Query { name: Name("wt.wikipedia.org."), query_type: A, query_class: IN }')

In [ ]:
from langchain_tavily import TavilySearch

tavily = TavilySearch(max_results=3)
resultat = tavily.invoke("dernières nouveautés de LangChain")

for r in resultat["results"]:
    print("-", r["title"])
    print("  ", r["url"])

In [ ]:
from langchain_experimental.tools import PythonREPLTool

python_repl = PythonREPLTool()
print(python_repl.invoke("print(sum(i**2 for i in range(1, 101)))"))

In [3]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=llm,
    tools=[duckduckgo, tavily, python_repl],
    system_prompt=(
        "Tu es un assistant de recherche. "
        "Pour les informations récentes, utilise la recherche web (Tavily ou DuckDuckGo). "
        "Pour tout calcul, écris et exécute du code Python avec print(). "
        "Réponds en français, brièvement."
    ),
)

def demander(question):
    print(f"\nQuestion : {question}")
    resultat = agent.invoke({"messages": [HumanMessage(question)]})
    for message in resultat["messages"]:
        if type(message).__name__ == "AIMessage" and message.tool_calls:
            for appel in message.tool_calls:
                print(f"  🔧 {appel['name']}({str(appel['args'])[:80]})")
    print(f"Réponse : {resultat['messages'][-1].content}")
    return resultat

NameError: name 'tavily' is not defined

In [4]:
demander("Quelles sont les dernières actualités sur l'intelligence artificielle ?")
demander("Calcule la somme des carrés des nombres de 1 à 100.")
demander("Quelle est la population du Maroc ? Calcule ensuite combien cela fait par km² sachant que la superficie est de 446 550 km².")

NameError: name 'demander' is not defined